# 04 · Protein interaction network

**Question:** How do the macrocephaly genes connect at the protein level? Which genes are **hubs**, and do the pathway groups found in notebook 03 (growth signalling vs metabolic/structural) form **separate modules**?

**Data:** [STRING](https://string-db.org) human protein–protein interactions, high confidence only (combined score ≥ 0.7).

**Input:** `results/` from notebooks 01–03  
**Outputs:** `results/network_edges.csv`, `results/network_nodes.csv`, `results/macrocephaly_network.graphml` (opens in Cytoscape), `figures/network.png`, `figures/hub_genes.png`

## 1. Setup
Upload `results_03.zip` through the Files panel first, then run.

In [ ]:
!unzip -q -o results_03.zip

import os, io, time
from datetime import date
import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

def read_set(path):
    with open(path) as f:
        return sorted({line.strip() for line in f if line.strip()})

macro = read_set("results/gene_sets/macrocephaly.txt")
shared = pd.read_csv("results/shared_genes.csv")
shared37 = set(shared.loc[shared["n_other_phenotypes"] == 3, "gene"])
print(f"Macrocephaly genes: {len(macro)} | shared with all three phenotypes: {len(shared37)}")

## 2. Download interactions from STRING
The STRING API returns only interactions **between the genes we send**, so the network contains macrocephaly genes only.  
`required_score=700` keeps high-confidence interactions (STRING scores run 0–1000).

In [ ]:
STRING_API = "https://string-db.org/api/tsv/network"
params = {
    "identifiers": "\r".join(macro),
    "species": 9606,              # human
    "required_score": 700,        # high confidence
    "caller_identity": "macrocephaly_gene_landscape",
}
r = requests.post(STRING_API, data=params, timeout=120)
r.raise_for_status()
raw = pd.read_csv(io.StringIO(r.text), sep="\t")
RUN_DATE = date.today().isoformat()

edges = (raw[["preferredName_A", "preferredName_B", "score"]]
         .rename(columns={"preferredName_A": "gene_A", "preferredName_B": "gene_B"})
         .drop_duplicates())
# STRING lists each pair once per direction in some versions; keep one row per pair
edges["pair"] = edges.apply(lambda x: tuple(sorted([x.gene_A, x.gene_B])), axis=1)
edges = edges.drop_duplicates("pair").drop(columns="pair")
edges.to_csv("results/network_edges.csv", index=False)

print(f"High-confidence interactions: {len(edges)} (STRING queried {RUN_DATE})")
edges.head()

## 3. Build the network
Genes with no high-confidence partner among the other macrocephaly genes are left out of the network. That is common and not an error: many disease genes act alone or with partners outside this set.

In [ ]:
G = nx.Graph()
G.add_weighted_edges_from(edges[["gene_A", "gene_B", "score"]].itertuples(index=False))

in_net = set(G.nodes)
not_mapped = sorted(set(macro) - in_net)
print(f"Genes in network: {G.number_of_nodes()} of {len(macro)}")
print(f"Interactions: {G.number_of_edges()}")
print(f"Genes with no high-confidence partner: {len(not_mapped)}")

components = sorted(nx.connected_components(G), key=len, reverse=True)
print(f"Connected components: {len(components)} | largest: {len(components[0])} genes")

## 4. Is the network more connected than chance?
If macrocephaly genes interact with each other more than random genes would, that suggests they work in shared pathways.  
STRING tests this directly with its **PPI enrichment p-value**, which compares the observed number of interactions with what a random set of proteins of the same size would have.

In [ ]:
r2 = requests.post("https://string-db.org/api/tsv/ppi_enrichment",
                   data={k: v for k, v in params.items()}, timeout=120)
r2.raise_for_status()
ppi = pd.read_csv(io.StringIO(r2.text), sep="\t")
ppi.to_csv("results/ppi_enrichment.csv", index=False)
ppi.T

**How to read this:** `number_of_edges` is what we observed, `expected_number_of_edges` is what random proteins of similar size would give. A very small `p_value` means the genes are much more connected than chance.

## 5. Hub genes
Two common ways to measure importance in a network:
- **Degree**: how many direct partners a gene has.
- **Betweenness centrality**: how often a gene lies on the shortest path between other genes. High betweenness means it acts as a bridge between parts of the network.

In [ ]:
deg = dict(G.degree())
btw = nx.betweenness_centrality(G)

nodes = pd.DataFrame({
    "gene": list(G.nodes),
    "degree": [deg[g] for g in G.nodes],
    "betweenness": [round(btw[g], 4) for g in G.nodes],
    "shared_all_three": [g in shared37 for g in G.nodes],
}).sort_values("degree", ascending=False).reset_index(drop=True)

nodes.head(15)

## 6. Modules
The **Louvain** method groups genes that are densely connected to each other into modules (communities).  
For each module, we list its largest-degree genes and the most specific enriched term from notebook 03 that it covers (at least 3 module genes, highest share of the term's genes). Broad terms like "signal transduction" are pushed down this way.

In [ ]:
modules = nx.community.louvain_communities(G, weight="weight", seed=42)
modules = sorted(modules, key=len, reverse=True)
mod_of = {g: i + 1 for i, m in enumerate(modules) for g in m}
nodes["module"] = nodes["gene"].map(mod_of)

enr = pd.read_csv("results/enrichment_macrocephaly.csv")
enr["gene_list"] = enr["intersections"].str.split(", ")

def best_term(module_genes, min_genes=3):
    # Most specific term covering the module: highest share of the term's genes
    # that fall in the module, with at least min_genes module genes in the term.
    best, best_n, best_score = "-", 0, 0
    for _, row in enr.iterrows():
        n = len(module_genes & set(row["gene_list"]))
        if n < min_genes:
            continue
        score = n / row["term_size"]
        if score > best_score or (score == best_score and n > best_n):
            best, best_n, best_score = f"{row['name']} ({row['source']})", n, score
    return best, best_n

summary = []
for i, m in enumerate(modules, start=1):
    if len(m) < 5:
        continue
    term, n = best_term(set(m))
    top = nodes[nodes["module"] == i].nlargest(5, "degree")["gene"].tolist()
    summary.append({"module": i, "size": len(m), "top_genes": ", ".join(top),
                    "best_matching_term": term, "genes_in_term": n})

module_summary = pd.DataFrame(summary)
module_summary.to_csv("results/module_summary.csv", index=False)
pd.set_option("display.max_colwidth", 80)
module_summary

**How to read this:** if the growth-signalling genes (PTEN, MTOR, HRAS…) and the peroxisomal or glycosylation genes (PEX…, POMT…) land in different modules, the network supports the two-route picture from notebook 03.

## 7. Save the network
`network_nodes.csv` is the gene table. The `.graphml` file opens directly in **Cytoscape** (File → Import → Network from File) for a nicer interactive figure.

In [ ]:
nodes.to_csv("results/network_nodes.csv", index=False)
for g in G.nodes:
    row = nodes.loc[nodes["gene"] == g].iloc[0]
    G.nodes[g]["degree"] = int(row["degree"])
    G.nodes[g]["betweenness"] = float(row["betweenness"])
    G.nodes[g]["module"] = int(row["module"])
    G.nodes[g]["shared_all_three"] = bool(row["shared_all_three"])
nx.write_graphml(G, "results/macrocephaly_network.graphml")

with open("results/network_run_info.txt", "w") as f:
    f.write(f"STRING queried: {RUN_DATE}\nSpecies: 9606 | required_score: 700\n")
    f.write(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()} | Modules (Louvain, seed 42): {len(modules)}\n")
print(open("results/network_run_info.txt").read())

## 8. Figure: the largest connected part of the network
Node colour = module, node size = number of partners, label = top hub of each module.

The short module names in `MODULE_NAMES` were written by hand after reading `module_summary.csv` from this run. If you rerun with a newer STRING version and the module numbers change, update this dictionary.

In [ ]:
MODULE_NAMES = {1: "PI3K/AKT/mTOR + Ras/MAPK", 2: "Wnt / beta-catenin", 3: "Chromatin (PRC2)",
                4: "Hedgehog", 5: "Cilia", 6: "Neurogenesis", 7: "Mixed", 8: "O-mannose glycosylation"}

H = G.subgraph(components[0]).copy()
pos = nx.spring_layout(H, seed=42, k=0.35, iterations=200)

big_modules = [i + 1 for i, m in enumerate(modules) if len(m & set(H.nodes)) >= 5][:8]
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3", "#8C8C8C"]
color_of = {m: palette[i] for i, m in enumerate(big_modules)}

fig, ax = plt.subplots(figsize=(11, 9))
nx.draw_networkx_edges(H, pos, alpha=0.12, width=0.5, ax=ax)
nx.draw_networkx_nodes(H, pos, node_color=[color_of.get(mod_of[g], "#D3D3D3") for g in H.nodes],
                       node_size=[15 + 8 * H.degree(g) for g in H.nodes],
                       edgecolors="white", linewidths=0.5, ax=ax)

# label the top hub of each coloured module; nudge two crowded labels apart
nudge = {"AKT1": (-0.09, 0.05), "CTNNB1": (0.08, -0.06)}
for m in big_modules:
    hub = nodes[(nodes["module"] == m) & nodes["gene"].isin(H.nodes)].nlargest(1, "degree")["gene"].iloc[0]
    x, y = pos[hub]
    dx, dy = nudge.get(hub, (0, 0.035))
    ax.annotate(hub, (x, y), (x + dx, y + dy), fontsize=9, ha="center", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color_of[m], lw=1),
                arrowprops=dict(arrowstyle="-", color="grey", lw=0.6) if hub in nudge else None)

for m in big_modules:
    ax.scatter([], [], color=color_of[m], s=60, label=f"{m}: {MODULE_NAMES.get(m, '')}")
ax.legend(loc="lower left", fontsize=8, frameon=False, title="Module", title_fontsize=9)
ax.set_title(f"Macrocephaly gene network (STRING >= 0.7): {H.number_of_nodes()} genes, "
             f"{H.number_of_edges()} interactions\nColour = module, node size = number of partners, "
             "label = top hub of each module", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.savefig("figures/network.png", dpi=200)
plt.show()

## 9. Figure: top hub genes

In [ ]:
top = nodes.nlargest(15, "degree").iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top["gene"], top["degree"],
        color=["#C44E52" if s else "#4C72B0" for s in top["shared_all_three"]])
ax.set_xlabel("Degree (high-confidence partners within the macrocephaly set)")
ax.set_title("Top 15 hub genes")
if top["shared_all_three"].any():
    ax.scatter([], [], color="#C44E52", marker="s", label="also linked to microcephaly, autism and seizure")
    ax.legend(fontsize=7, loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig("figures/hub_genes.png", dpi=200)
plt.show()

## 10. Limitations
- STRING mixes evidence types (experiments, databases, co-expression, text-mining). A high score does not always mean a direct physical interaction.
- Well-studied proteins have more recorded interactions, so hub status partly reflects research attention (the same **study bias** noted in notebook 02).
- Louvain modules can change slightly with the random seed; the seed is fixed at 42 for reproducibility.

## 11. Download the results

In [ ]:
!zip -qr results_04.zip results/ figures/
from google.colab import files
files.download("results_04.zip")